# 01 - BRONZE | Ingestao do catalogo de seguranca do Pitaia

**Etapa 4.2 do MVP (Coleta)**

Le os 8 CSVs extraidos do SQL Editor do Supabase e persiste como tabelas
Delta, **sem nenhuma transformacao de conteudo**.

## Decisoes de projeto

1. **Tudo entra como STRING** (`inferSchema=False`). A Bronze e o cofre de
   evidencias: se o Postgres devolveu `t` em vez de `true`, ou um array
   `{anon;authenticated}`, fica registrado assim. A tipagem acontece na Silver.
2. **Nenhum filtro de schema.** Os schemas internos do Supabase (`auth`,
   `storage`, `realtime`, `vault`) entram junto. Eles serao usados na analise
   como **grupo de comparacao** contra os schemas da aplicacao.
3. **Metadados de controle** em toda tabela: `_ingerido_em`, `_fonte`,
   `_snapshot`, `_arquivo_origem`.

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col
from datetime import date

CATALOGO   = "workspace"          # ajuste se criou catalogo proprio
SCH_BRONZE = "bronze"
VOLUME     = "raw"
SNAPSHOT   = date.today().isoformat()

VOL = f"/Volumes/{CATALOGO}/{SCH_BRONZE}/{VOLUME}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{SCH_BRONZE} "
          f"COMMENT 'Camada Bronze: catalogo de seguranca do Postgres/Supabase do Pitaia, "
          f"exatamente como extraido, sem transformacao.'")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{SCH_BRONZE}.{VOLUME}")

print(f"Suba os 8 CSVs em: {VOL}")

Suba os 8 CSVs em: /Volumes/workspace/bronze/raw


## Ingestao das 8 fontes

In [0]:
FONTES = {
    "objetos":     "objetos.csv",
    "politicas":   "politicas.csv",
    "colunas":     "colunas.csv",
    "grants":      "grants.csv",
    "funcoes":     "funcoes.csv",
    "buckets":     "buckets.csv",
    "extensoes":   "extensoes.csv",
    "constraints": "constraints.csv",
}

resumo = []

for tabela, arquivo in FONTES.items():
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)        # expressoes SQL das politicas tem \n
          .option("escape", '"')            # e virgulas dentro de aspas
          .option("inferSchema", False)     # Bronze = cru, tudo string
          .csv(f"{VOL}/{arquivo}")
          # input_file_name() nao e suportado no Unity Catalog; a coluna oculta
          # _metadata.file_path faz o mesmo papel de linhagem
          .withColumn("_arquivo_origem", col("_metadata.file_path"))
          .withColumn("_fonte", lit(f"supabase_sql_editor::{arquivo}"))
          .withColumn("_snapshot", lit(SNAPSHOT))
          .withColumn("_ingerido_em", current_timestamp()))

    destino = f"{CATALOGO}.{SCH_BRONZE}.{tabela}"
    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(destino))

    n = spark.table(destino).count()
    resumo.append((tabela, arquivo, n, len(df.columns)))
    print(f"  {destino:45s}  {n:6d} linhas  {len(df.columns)} colunas")

  workspace.bronze.objetos                          344 linhas  11 colunas
  workspace.bronze.politicas                        209 linhas  12 colunas
  workspace.bronze.colunas                         3311 linhas  11 colunas
  workspace.bronze.grants                          5707 linhas  9 colunas
  workspace.bronze.funcoes                          195 linhas  9 colunas
  workspace.bronze.buckets                            7 linhas  9 colunas
  workspace.bronze.extensoes                          8 linhas  7 colunas
  workspace.bronze.constraints                      380 linhas  8 colunas


## Evidencia de persistencia (screenshot deste resultado para o README)

In [0]:
df_resumo = spark.createDataFrame(
    resumo, "tabela string, arquivo_origem string, linhas int, colunas int")
display(df_resumo.orderBy("tabela"))

tabela,arquivo_origem,linhas,colunas
buckets,buckets.csv,7,9
colunas,colunas.csv,3311,11
constraints,constraints.csv,380,8
extensoes,extensoes.csv,8,7
funcoes,funcoes.csv,195,9
grants,grants.csv,5707,9
objetos,objetos.csv,344,11
politicas,politicas.csv,209,12


## Documentacao das tabelas Bronze no Unity Catalog

In [0]:
COMENTARIOS = {
 "objetos": "Um registro por objeto do banco (tabela, view, materializada). Traz o estado do Row Level Security e as reloptions. Fonte: pg_class + pg_namespace.",
 "politicas": "Um registro por politica de RLS. Coluna roles vem como array serializado com ';'. expressao_using e NULL em politicas que so definem WITH CHECK. Fonte: pg_policies.",
 "colunas": "Um registro por coluna de cada objeto. Base para a classificacao de dado pessoal e de saude (LGPD). Fonte: pg_attribute + pg_class. Nao usa information_schema.columns porque aquela view e filtrada por privilegio e devolve subconjunto silencioso.",
 "grants": "Um registro por combinacao role x objeto x privilegio. Maior volume da extracao. Fonte: aclexplode(pg_class.relacl). Nao usa information_schema.role_table_grants porque aquela view so mostra grants da role corrente - na primeira extracao devolveu 164 linhas de uma unica role em vez de 5707.",
 "funcoes": "Um registro por funcao. Somente FLAGS (security_definer, config); o corpo da funcao NAO foi extraido por decisao de seguranca. Fonte: pg_proc.",
 "buckets": "Um registro por bucket do Supabase Storage. publico=true significa leitura anonima. Fonte: storage.buckets.",
 "extensoes": "Um registro por extensao instalada. Cadeia de suprimentos, OWASP A03:2025. Fonte: pg_extension.",
 "constraints": "Um registro por constraint de PK, UNIQUE ou FK. Fonte: pg_constraint + pg_class, pelo mesmo motivo de filtro por privilegio.",
}

def sql_txt(t):
    """Aspa simples dentro do texto quebra o literal SQL. Dobrar e o escape do Postgres/Spark."""
    return t.replace("'", "''")

for tabela, txt in COMENTARIOS.items():
    spark.sql(f"COMMENT ON TABLE {CATALOGO}.{SCH_BRONZE}.{tabela} IS '{sql_txt(txt)}'")

for tabela in FONTES:
    t = f"{CATALOGO}.{SCH_BRONZE}.{tabela}"
    spark.sql(f"ALTER TABLE {t} ALTER COLUMN _ingerido_em COMMENT 'Timestamp da carga na Bronze. Metadado de controle.'")
    spark.sql(f"ALTER TABLE {t} ALTER COLUMN _fonte COMMENT 'Identificacao da fonte e do arquivo de origem. Linhagem.'")
    spark.sql(f"ALTER TABLE {t} ALTER COLUMN _snapshot COMMENT 'Data do snapshot do catalogo de seguranca (AAAA-MM-DD).'")

print("Comentarios aplicados. Screenshot do Catalog Explorer agora.")

Comentarios aplicados. Screenshot do Catalog Explorer agora.
